# Mini Projet — Partie 1 : Data Process + Data Visualization
## « L'Afrique de l'Ouest est-elle bien desservie par le transport aérien ? »

**UE FDIA — Traitement de données avec Python** | CITADEL / UVBF

---

### Objectif

Ce projet combine deux compétences vues en cours :
- **Phase A (Data Process)** : import, nettoyage, filtrage et extraction de données avec Pandas (séquences 1 à 7 et 12).
- **Phase B (Data Visualization)** : transformer un résultat de la Phase A en une communication explicative claire, en appliquant les principes du cours (contexte Who/What/How, choix du graphique, élimination du clutter).

### Données

Source ouverte et accessible : [OurAirports](https://ourairports.com/data/), base mondiale des aéroports, mise à jour quotidiennement.

```
URL = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"
```

### Consignes générales

- Complétez toutes les cellules marquées `# TODO` ou `*(à compléter)*`.
- Ne supprimez pas les cellules de justification Markdown : elles font partie de l'évaluation.
- Le notebook doit pouvoir être exécuté de haut en bas sans erreur (`Restart & Run All`).
- Committez votre travail progressivement sur GitHub (un seul commit final = pénalité).

### Rendu attendu

1. Ce notebook complété (Jupyter ou Google Colab), code et sorties visibles.
2. Un dépôt GitHub `mini_projet_data_dataviz_partie1` contenant : ce notebook, les fichiers CSV produits, l'image du graphique final, et un fichier `storyboard.md`.

---
# Étape 0 — Identification et individualisation des paramètres

Chaque étudiant reçoit un **sous-ensemble de pays** et un **scénario de communication** différents, déterminés automatiquement à partir du matricule. Cela garantit que chaque projet repose sur des données et un contexte propres à l'étudiant.

**Remplacez `A_COMPLETER` ci-dessous par votre matricule étudiant, puis exécutez la cellule.**

In [ ]:
import hashlib

MATRICULE = "N05244720212"

REGIONS = {
    0: ("Afrique de l'Ouest", ["BF", "ML", "NE", "SN", "CI", "GH", "NG", "TG", "BJ", "GN"]),
    1: ("Afrique de l'Est", ["KE", "TZ", "UG", "ET", "RW", "SO", "DJ", "BI"]),
    2: ("Afrique australe", ["ZA", "ZW", "ZM", "BW", "NA", "MZ", "MW"]),
    3: ("Amérique centrale", ["GT", "HN", "SV", "NI", "CR", "PA", "BZ"]),
    4: ("Asie du Sud-Est", ["TH", "VN", "KH", "LA", "MM", "PH", "ID", "MY"]),
    5: ("Europe de l'Est", ["PL", "CZ", "SK", "HU", "RO", "BG", "UA"]),
}

SCENARIOS = {
    0: ("A", "Comité d'investissement régional (ex. BOAD)",
        "Justifier un investissement dans un pays sous-desservi de votre échantillon"),
    1: ("B", "Grand public, via un article de blog",
        "Sensibiliser sur les inégalités d'accès au transport aérien dans votre échantillon"),
    2: ("C", "Compagnie aérienne low-cost",
        "Identifier une opportunité de marché : un pays sans grand aéroport actif"),
}

assert MATRICULE != "A_COMPLETER", "Renseignez votre matricule avant de continuer."

seed = int(hashlib.sha256(MATRICULE.encode()).hexdigest(), 16)
region_id = seed % len(REGIONS)
scenario_id = (seed // len(REGIONS)) % len(SCENARIOS)

region_name, region_countries = REGIONS[region_id]
scenario_code, scenario_who, scenario_what = SCENARIOS[scenario_id]

print(f"Région assignée   : {region_name} {region_countries}")
print(f"Scénario assigné  : {scenario_code}")
print(f"  Who (audience)  : {scenario_who}")
print(f"  What (message)  : {scenario_what}")

---
# PHASE A — Data Process

## A.1 — Import des données

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"

df_raw = pd.read_csv(URL)
print(f"Dimensions du dataset brut : {df_raw.shape}")
df_raw.head()

## A.2 — Prétraitement

In [ ]:
# Sélection et renommage des colonnes utiles
cols_mapping = {
    "id": "ID",
    "name": "airport_name",
    "type": "type",
    "iso_country": "country",
    "latitude_deg": "lat",
    "longitude_deg": "long",
    "elevation_ft": "elevation",
    "scheduled_service": "service",
}

df = df_raw[list(cols_mapping.keys())].rename(columns=cols_mapping)
print(f"Dimensions après sélection des colonnes : {df.shape}")
print(f"Colonnes conservées : {list(df.columns)}")
df.head()

In [ ]:
# Analyse des valeurs manquantes dans la colonne elevation
nb_nan = df["elevation"].isnull().sum()
pct_nan = nb_nan / len(df) * 100
print(f"Valeurs manquantes dans 'elevation' : {nb_nan} sur {len(df)} lignes ({pct_nan:.1f}%)")

print("\nStratégie appliquée : conservation des lignes — aucune suppression.")
print(f"Le DataFrame reste à {df.shape[0]} lignes.")

**Justification de la stratégie choisie :**

La colonne `elevation` présente environ 17 % de valeurs manquantes dans le dataset global. Deux approches alternatives auraient pu être envisagées : (1) supprimer toutes les lignes concernées, ou (2) imputer par la médiane.

La suppression est écartée parce que l'altitude n'est pas la variable d'intérêt principale de ce projet — l'analyse cible le type d'aéroport (`type`) et la présence d'un service régulier (`service`). Retirer 14 000 lignes risquerait de déformer la représentation géographique de certains pays, notamment ceux qui n'ont que des petits aéroports souvent non renseignés en altitude. L'imputation par la médiane est également écartée : l'élévation varie considérablement selon la géographie (zones côtières vs plateaux montagneux d'Asie du Sud-Est) et une valeur globale unique n'aurait aucun sens physique.

La stratégie retenue est donc de **conserver les NaN** dans le DataFrame propre. Lors des seules opérations qui requièrent l'altitude (`nsmallest` / `nlargest`), un `dropna(subset=["elevation"])` sera appliqué localement, sans altérer le reste de l'analyse.

In [ ]:
# Sauvegarde du fichier nettoyé
df.to_csv("airports_clean.csv", index=False)
print("Fichier 'airports_clean.csv' sauvegardé :", df.shape)

## A.3 — Opérations individualisées

Toutes les opérations suivantes portent uniquement sur la **région Asie du Sud-Est** assignée à l'Étape 0 : TH, VN, KH, LA, MM, PH, ID, MY.

In [ ]:
# Filtrage sur les pays de la région assignée
df_region = df[df["country"].isin(region_countries)].copy()

print(f"Nombre d'aéroports dans la région {region_name} : {len(df_region)}")
print("\nRépartition par pays :")
print(df_region["country"].value_counts())
df_region.head()

In [ ]:
# 1. Extraction des aéroports dont le nom contient le mot-clé « International »
international = df_region[
    df_region["airport_name"].str.contains("International", case=False, na=False)
].copy()
international.to_csv("international.csv", index=False)
print(f"Aéroports avec 'International' dans le nom : {len(international)}")
print(international[["airport_name", "country", "type", "service"]].to_string())

In [ ]:
# 2. Extraction des aéroports dont le nom commence entre A et K (plage alphabétique)
range_alpha = df_region[
    df_region["airport_name"].str[0].str.upper().between("A", "K")
].copy()
range_alpha.to_csv("range_alpha.csv", index=False)
print(f"Aéroports dont le nom commence entre A et K : {len(range_alpha)}")
range_alpha[["airport_name", "country"]].head(10)

In [ ]:
# 3. Aéroport à l'altitude minimale et maximale de la région
df_region_elev = df_region.dropna(subset=["elevation"])

aero_min = df_region_elev.nsmallest(1, "elevation")[["airport_name", "country", "elevation"]]
aero_max = df_region_elev.nlargest(1, "elevation")[["airport_name", "country", "elevation"]]

print("Aéroport à l'altitude minimale :")
print(aero_min.to_string(index=False))

print("\nAéroport à l'altitude maximale :")
print(aero_max.to_string(index=False))

### Synthèse par pays

In [ ]:
# Table de synthèse : nombre total, large_airport, service régulier par pays
synthese = df_region.groupby("country").agg(
    nb_aeroports=("airport_name", "count"),
    nb_large=("type", lambda x: (x == "large_airport").sum()),
    nb_service_regulier=("service", lambda x: (x == "yes").sum())
).reset_index()

print(f"Table de synthèse par pays — {region_name} :")
synthese

In [ ]:
# Identification des pays sans aucun grand aéroport en service régulier
df_large_actif = df_region[
    (df_region["type"] == "large_airport") & (df_region["service"] == "yes")
]
pays_avec_grand = set(df_large_actif["country"].unique())
pays_region = set(region_countries)
pays_sous_desservis = pays_region - pays_avec_grand

print("Pays avec au moins un grand aéroport actif :", sorted(pays_avec_grand))
print("Pays sous-desservis (aucun grand aéroport actif) :",
      sorted(pays_sous_desservis) if pays_sous_desservis else "Aucun — tous les pays ont au moins un.")

synthese.to_csv("synthese_pays.csv", index=False)
print("\nFichier 'synthese_pays.csv' sauvegardé.")

---
# PHASE B — Data Visualization

Cette phase part directement des résultats de la Phase A (table `synthese`) pour construire une visualisation explicative, en suivant la méthode vue en cours.

## B.1 — Contexte (Who / What / How)

**Scénario assigné : A — Comité d'investissement régional**

- **Who (audience)** : Les membres d'un comité d'investissement régional spécialisé en infrastructure de transport en Asie du Sud-Est (équivalent de la BOAD pour la région). Ce sont des décideurs techniques et financiers habitués aux données chiffrées, sensibles aux arguments de rentabilité économique et de connectivité régionale. Ils attendent des preuves concrètes, pas des généralités.

- **What (message / action attendue)** : Démontrer que le Cambodge (KH) est le pays d'Asie du Sud-Est le plus sous-dimensionné en matière de connectivité aérienne commerciale : seulement 5 aéroports en service régulier sur 21 recensés, soit le ratio le plus faible de la région. L'objectif est d'obtenir un mandat d'étude de faisabilité pour un investissement dans le développement des infrastructures aéroportuaires au Cambodge.

- **How (mécanisme)** : Présentation orale de 10 minutes lors d'une séance du Comité, appuyée par ce graphique épuré. Le ton est sobre, factuel, orienté décision. On laisse les données parler.

- **Ton souhaité** : Professionnel, direct, factuel. Le graphique doit permettre une lecture immédiate sans effort d'interprétation de la part de l'audience.

## B.2 — Exploratoire vs Explicatoire

**Analyse exploratoire :**

En parcourant la table `synthese`, plusieurs contrastes apparaissent immédiatement. L'Indonésie (ID) domine très largement la région sur tous les indicateurs : 728 aéroports au total, 19 grands aéroports, 156 en service régulier — soit plus que tous les autres pays réunis. C'est une anomalie liée à l'immensité du territoire (17 000 îles), qui risque d'écraser visuellement les autres pays si elle est incluse sans précaution dans un graphique.

En excluant ce cas extrême, les disparités restent marquées. Les Philippines (PH) arrivent en deuxième position avec 45 liaisons actives, suivies de la Malaisie (MY) et de la Thaïlande (TH) à 34 chacune. À l'opposé, le Cambodge (KH) n'a que 5 aéroports en service régulier sur 21 recensés — le chiffre le plus bas de la région. Le Laos (LA) vient juste après avec 7 services réguliers sur 25 aéroports.

Un autre fait notable : le Cambodge possède 5 grands aéroports (le plus fort ratio grands aéroports / total de la région avec KH), mais seulement 5 en service régulier — ce qui suggère que l'infrastructure existe mais n'est pas pleinement exploitée commercialement.

**Analyse explicative :**

Le message retenu est : **le nombre d'aéroports en service régulier par pays révèle de profondes inégalités de connectivité commerciale en Asie du Sud-Est.** Le Cambodge est mis en évidence comme le pays avec le moins de liaisons commerciales actives en valeur absolue. Ce message est choisi parce qu'il répond directement à la question d'investissement du scénario A, et parce que la variable `nb_service_regulier` est la plus directement liée à la connectivité économique réelle d'un territoire.

## B.3 — Storyboard

*(Repris également dans le fichier `storyboard.md` à la racine du dépôt.)*

1. **Issue** : La connectivité aérienne commerciale en Asie du Sud-Est est très inégalement répartie entre les pays de la région, y compris entre pays de taille comparable.

2. **Démonstration de l'issue** : Sur 8 pays analysés, l'Indonésie concentre à elle seule 156 des 326 aéroports en service régulier (48%). Les 7 autres pays se partagent les 170 restants de façon très inégale. Le Cambodge, avec seulement 5 liaisons actives, se retrouve en dernière position — alors qu'il dispose de 5 grands aéroports recensés dans la base.

3. **Idée retenue** : Le Cambodge est le pays de la région présentant le plus faible nombre d'aéroports en service régulier (5), alors que son parc de grands aéroports (5 également) justifierait une bien meilleure couverture commerciale. Ce décalage entre infrastructure existante et service effectif est une opportunité d'investissement directe.

4. **Recommandation** : Le comité devrait mandater une étude de faisabilité pour le développement des liaisons commerciales au Cambodge, en ciblant en priorité les connexions intra-régionales vers les hubs de Bangkok, Kuala Lumpur et Manille.

## B.4 — Avant : graphique par défaut

In [ ]:
import matplotlib.pyplot as plt

# Variable choisie : nb_service_regulier — directement liée à la connectivité commerciale
colonne_a_visualiser = "nb_service_regulier"

fig, ax = plt.subplots()
ax.bar(synthese["country"], synthese[colonne_a_visualiser])
ax.set_title(f"{colonne_a_visualiser} by country")
plt.show()

## B.5 — Choix du graphique (justification)

Le cours recommande le **bar chart horizontal** pour comparer une quantité numérique entre catégories (pays). Ce choix est justifié par plusieurs raisons concrètes :
- Les étiquettes pays se lisent naturellement de haut en bas, sans rotation forcée.
- L'ordre de tri des barres (du plus petit au plus grand) permet une lecture immédiate du classement.
- La longueur des barres encode une quantité sur un axe commun, ce que l'œil humain compare avec précision.

**Pourquoi rejeter pie chart, donut chart et graphique 3D :**

- **Pie / donut** : Ces formats encodent la donnée en angle ou en arc de cercle. La perception humaine est mauvaise pour comparer des angles — il est difficile de dire à l'œil si une part représente 12% ou 15% sans lire les étiquettes. Avec 8 pays, le graphique en camembert devient illisible. Ces formats ne conviennent que pour 2-3 catégories, et uniquement quand la part-du-tout est le message central.

- **Graphique 3D** : La troisième dimension n'encode aucune variable supplémentaire ici. Elle introduit une distorsion de perspective (les barres à l'arrière-plan paraissent plus courtes), ce qui fausse les comparaisons. C'est du clutter pur : visuellement complexe, informationnellement vide.

## B.6 — Decluttering pas à pas

In [ ]:
# Étape 1 — Passage en barres horizontales + suppression des bordures

synthese_sorted = synthese.sort_values(colonne_a_visualiser)

fig, ax = plt.subplots()
ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser])

# Suppression des 4 bordures
for spine in ["top", "right", "bottom", "left"]:
    ax.spines[spine].set_visible(False)

plt.show()

In [ ]:
# Étape 2 — Nettoyage des axes

fig, ax = plt.subplots()
ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser])

for spine in ["top", "right", "bottom", "left"]:
    ax.spines[spine].set_visible(False)

# Suppression des graduations et label de l'axe x
ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.set_xlabel("")
ax.tick_params(axis="y", length=0)

plt.show()

In [ ]:
# Étape 3 — Labellisation directe + couleur intentionnelle (mise en évidence du Cambodge)

pays_a_mettre_en_evidence = "KH"  # Cambodge : pays le moins desservi de la région

# Couleurs : orange pour KH, gris pour tous les autres
couleurs = [
    "#E07B39" if pays == pays_a_mettre_en_evidence else "#CCCCCC"
    for pays in synthese_sorted["country"]
]

fig, ax = plt.subplots()
bars = ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color=couleurs)

for spine in ["top", "right", "bottom", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.set_xlabel("")
ax.tick_params(axis="y", length=0)

# Label direct au bout de chaque barre
for bar, val in zip(bars, synthese_sorted[colonne_a_visualiser]):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(int(val)),
        va="center", ha="left", fontsize=10
    )

plt.show()

In [ ]:
# Étape 4 — Version finale : titre déclaratif portant l'insight

titre_declaratif = "Le Cambodge, pays d'Asie du Sud-Est avec le moins d'aéroports en service régulier"

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color=couleurs)

for spine in ["top", "right", "bottom", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.set_xlabel("")
ax.tick_params(axis="y", length=0)

for bar, val in zip(bars, synthese_sorted[colonne_a_visualiser]):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(int(val)),
        va="center", ha="left", fontsize=10
    )

ax.set_title(titre_declaratif, fontsize=13, fontweight="bold", loc="left")
plt.tight_layout()
plt.savefig("graphique_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé : graphique_final.png")

## B.7 — Message final

En présentant ces données au comité d'investissement, l'objectif est de montrer que parmi les 8 pays d'Asie du Sud-Est étudiés, le Cambodge (KH) est celui dont la connectivité aérienne commerciale est la plus faible en valeur absolue : seulement 5 aéroports en service régulier, alors que la base de données recense 5 grands aéroports sur son territoire. Ce décalage entre infrastructure existante et service effectif indique non pas une absence d'outil, mais une sous-exploitation commerciale — ce qui rend le cas cambodgien particulièrement pertinent pour une intervention ciblée.

Le graphique montre également que le Laos (LA) présente un profil similaire (7 services réguliers), mais le Cambodge cumule le chiffre absolu le plus bas de toute la région. La recommandation est de mandater une étude de faisabilité pour le développement des liaisons commerciales au Cambodge, en ciblant en priorité les connexions intra-régionales vers les grands hubs de la zone (Bangkok, Kuala Lumpur, Manille).

---
## Rappel — Rendu attendu

1. Ce notebook complété (toutes les cellules exécutées, sorties visibles).
2. `airports_clean.csv`, `synthese_pays.csv`, `graphique_final.png`.
3. `storyboard.md` (reprise de la section B.3).
4. Dépôt GitHub `mini_projet_data_dataviz_partie1` avec un historique de commits incrémental.